## Load Dependencies

In [22]:
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, trim_messages
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_history_aware_retriever

from langchain_huggingface import HuggingFaceEmbeddings

import bs4
from dotenv import load_dotenv
load_dotenv()

True

## LLM Setup

In [2]:
## Load API key
groq_api_key = os.getenv("GROQ_API_KEY")

## Load LLM
llm = ChatGroq(model="Llama3-8b-8192", groq_api_key=groq_api_key)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001969FFF6830>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001969FFF43A0>, model_name='Llama3-8b-8192', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

## Load, chunk and index the contents of the blog for Retriever

In [12]:
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2024-07-07-hallucination/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    )
)
docs = loader.load()
docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2024-07-07-hallucination/'}, page_content='\n\n      Extrinsic Hallucinations in LLMs\n    \nDate: July 7, 2024  |  Estimated Reading Time: 29 min  |  Author: Lilian Weng\n\n\nHallucination in large language models usually refers to the model generating unfaithful, fabricated, inconsistent, or nonsensical content. As a term, hallucination has been somewhat generalized to cases when the model makes mistakes. Here, I would like to narrow down the problem of hallucination to cases where the model output is fabricated and not grounded by either the provided context or world knowledge.\nThere are two types of hallucination:\n\nIn-context hallucination: The model output should be consistent with the source content in context.\nExtrinsic hallucination: The model output should be grounded by the pre-training dataset. However, given the size of the pre-training dataset, it is too expensive to retrieve and identify conflicts per g

In [13]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retreiver = vectorstore.as_retriever()
retreiver

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000196B7A0E320>, search_kwargs={})

## Prompt Template

In [14]:
system_prompt = (
    "You are a helpful and concise assistant for question-answering tasks. "
    "Use the retrieved context provided below to answer the user's question. "
    "If the answer is not contained within the context, respond with 'I don't know.' "
    "Limit your response to a maximum of three clear and concise sentences."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

#### Creating Chain

Using built-in chain constructors: `create_stuff_documents_chain`

In [30]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retreiver, question_answer_chain)

In [19]:
response = rag_chain.invoke({"input":"What is Sampling-Based Detection?"})
response

{'input': 'What is Sampling-Based Detection?',
 'context': [Document(id='6e0ec077-ee24-428b-a503-6f0a56e898fb', metadata={'source': 'https://lilianweng.github.io/posts/2024-07-07-hallucination/'}, page_content='Sampling Methods#\nLee, et al. (2022) found that nucleus sampling (top-$p$ sampling) is found to perform worse on FactualityPrompt benchmark than greedy sampling, although it achieves better diversity and less repetition, since nucleus sampling added extra randomness. So they proposed factual-nucleus sampling algorithm, based on the hypothesis that sampling randomness does more harm to factuality at the latter part of the sentence than at the beginning. Factual-nucleus sampling is designed to dynamically adapt the probability $p$ during sampling tokens for each sentence. For the $t$-th token in one sentence, we have $p_t = \\max(\\omega, p \\cdot \\lambda^{t−1})$ where $\\omega$ is to prevent the sampling falls back to greedy that hurts generation quality and diversity.'),
  Doc

In [21]:
response['answer']

'Sampling-Based Detection is a method used to detect factuality mistakes in language models, specifically by relying on consistency checks on factuality mistakes against multiple samples from a black-box LLM (Large Language Model).'

In [20]:
rag_chain.invoke({"input":"How do we achieve it ?"})

{'input': 'How do we achieve it ?',
 'context': [Document(id='7c77ff1b-c9f3-4c59-b2b3-79484f68e9f1', metadata={'source': 'https://lilianweng.github.io/posts/2024-07-07-hallucination/'}, page_content='Fine-tuning New Knowledge#\nFine-tuning a pre-trained LLM via supervised fine-tuning and RLHF is a common technique for improving certain capabilities of the model like instruction following. Introducing new knowledge at the fine-tuning stage is hard to avoid.\nFine-tuning usually consumes much less compute, making it debatable whether the model can reliably learn new knowledge via small-scale fine-tuning. Gekhman et al. 2024 studied the research question of whether fine-tuning LLMs on new knowledge encourages hallucinations. They found that (1) LLMs learn fine-tuning examples with new knowledge slower than other examples with knowledge consistent with the pre-existing knowledge of the model; (2) Once the examples with new knowledge are eventually learned, they increase the model’s tendenc

We can clearly see that `rag_chain` is not able to understand the context. I was asking about Sampling-Based Detection and it returned me the topic of fine-tuning new knowledge.

## Adding Chat History

In [24]:
contextualise_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualise_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualise_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [25]:
history_aware_retriever = create_history_aware_retriever(llm, retreiver, contextualise_q_prompt)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000196B7A0E320>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessag

#### Creating Chain

In [31]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [33]:
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(retreiver, question_answer_chain)

#### Appending chat history

In [34]:
chat_history=[]
question1 = "What is Sampling-Based Detection?"
response1 = rag_chain.invoke({"input":question1, "chat_history":chat_history})

chat_history.extend(
    [
        HumanMessage(content=question1),
        AIMessage(content=response1["answer"])
    ]
)

question2 = "How do we achieve it?"
response2 = rag_chain.invoke({"input":question2, "chat_history":chat_history})

print(response2['answer'])

I don't know. The provided context does not mention the specific steps or methods to achieve Sampling-Based Detection.
